In [0]:
%pip install google-search-results deltalake
dbutils.library.restartPython()

In [0]:
# Re-import packages after Python restart
import pandas as pd
import pyarrow as pa
from datetime import datetime
from serpapi import GoogleSearch
from deltalake.writer import write_deltalake

In [0]:


# Set credentials and config - REPLACE WITH YOUR 
# Get from serpapi.com
FLIGHT_OUTBOUND = "2026-05-01"  
FLIGHT_RETURN = "2026-05-08"
Flight_Currency = "USD"
google_flights = "google_flights"
# ACTUAL SERPAPI KEY
SERPAPI_KEY = dbutils.secrets.get(scope="my-api-scope", key="serpapi-key") 

# Define schema for flight data using pyarrow
SCHEMA = pa.schema([
    ("flight_number", pa.string()),
    ("airline", pa.string()),
    ("origin", pa.string()),
    ("destination", pa.string()),
    ("departure_time", pa.string()),
    ("arrival_time", pa.string()),
    ("duration", pa.string()),
    ("stops", pa.int64()),
    ("price", pa.float64()),
    ("currency", pa.string()),
    ("trip_type", pa.string()),
    ("fetched_at", pa.string()),
])

DC_AIRPORTS = ["IAD","DCA","BWI"] # Washington DC area airports
FL_AIRPORTS = ["MCO","MIA","FLL","TPA","RSW"] # Florida airports




In [0]:
def extract_flights(origin, destination):
    """Fetch flights for a single origin-destination pair"""
    params = {
        "engine": "google_flights",
        "departure_id": origin,
        "arrival_id": destination,
        "outbound_date": FLIGHT_OUTBOUND,
        "return_date": FLIGHT_RETURN,
        "currency": Flight_Currency,
        "hl": "en",
        "deep_search": "true",
        "api_key": SERPAPI_KEY,
    }

    search = GoogleSearch(params)
    results = search.get_dict()
    
    if "error" in results:
        print(f"[{origin} → {destination}] error: {results['error']}")
        return pd.DataFrame()

    flights = []
    fetched_at = datetime.utcnow().isoformat()

    for key in ["best_flights", "other_flights"]:
        for offer in results.get(key) or []:
            segs = offer.get("flights", [])
            if not segs:
                continue
            first = segs[0]
            last = segs[-1]
            flights.append({
                "flight_number": first.get("flight_number"),
                "airline": first.get("airline"),
                "origin": first.get("departure_airport", {}).get("id"),
                "destination": last.get("arrival_airport", {}).get("id"),
                "departure_time": first.get("departure_airport", {}).get("time"),
                "arrival_time": last.get("arrival_airport", {}).get("time"),
                "duration": str(offer.get("total_duration")),
                "stops": int(len(offer.get("layovers") or [])),
                "price": float(offer.get("price") or 0),
                "currency": Flight_Currency,
                "trip_type": "outbound",
                "fetched_at": fetched_at,
            })
    return pd.DataFrame(flights)  # ADD THIS LINE

def write_to_delta_table(df, table_name):
    """Writes pandas df to Delta table using Spark"""
    if df.empty:
        print(f"⚠️ Skipping {table_name} - no data")
        return
    
    # Convert pandas DataFrame to Spark DataFrame
    spark_df = spark.createDataFrame(df)
    
    # Write to Delta table (table_name must be quoted string)
    spark_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(table_name)
    
    print(f"✓ Wrote {len(df)} rows to {table_name}")


def extract_all_flights(origins, destinations):
    """Fetch flights for all origin-destination combinations"""
    all_flights = []
    
    for origin in origins:
        for destination in destinations:
            df = extract_flights(origin, destination)
            if not df.empty:
                all_flights.append(df)
    
    if all_flights:
        combined = pd.concat(all_flights, ignore_index=True)
        print(f"\n✓ Total flights fetched: {len(combined)}")
        write_to_delta_table(combined, "workspace.bronze.Airflights_Outbound")
        return combined
    else:
        print("⚠️ No flights found")
        return pd.DataFrame()


# Fetch all DC → FL flights (3 x 5 = 15 combinations)
df = extract_all_flights(DC_AIRPORTS, FL_AIRPORTS)

display(df)

In [0]:
%sql
select* from workspace.bronze.Airflights_Outbound
